# 11: Benchmarks

LLM inference benchmark: pp512 (prefill) and tg128 (decode).

In [ ]:
import os
import sys

if "COLAB_GPU" in os.environ:
    os.system("pip install -q git+https://github.com/PritRaj1/smelt.git")
    os.system("pip install -q 'transformers>=4.51,<5' psutil matplotlib")
else:
    sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd())))

import gc
import platform
import time

import matplotlib.pyplot as plt
import psutil
import torch
import torch.utils.benchmark as bench
from transformers import AutoModelForCausalLM, AutoTokenizer

import smelt

In [ ]:
MODEL_ID = "microsoft/bitnet-b1.58-2B-4T"
PP = 512
TG = 128

HAS_CUDA = torch.cuda.is_available()
cpu_name = platform.processor() or "CPU"
gpu_label = torch.cuda.get_device_name(0) if HAS_CUDA else "N/A"
n_threads = psutil.cpu_count(logical=False)
torch.set_num_threads(n_threads)

print(f"CPU: {cpu_name} ({n_threads} cores)")
print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")
if HAS_CUDA:
    print(f"GPU: {gpu_label}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Kernel microbenchmarks

Ternary GEMM vs torch.mm at BitNet 2B layer sizes.

In [ ]:
from smelt._clib import load_lib
from smelt.matmul import pack_tl1, quantize_ternary

lib = load_lib()

sizes = [
    ("attn_proj", 1, 2560, 2560),
    ("gate/up", 1, 2560, 6912),
    ("down", 1, 6912, 2560),
    ("lm_head", 1, 2560, 32768),
    ("prefill_16", 16, 2560, 2560),
    ("prefill_128", 128, 2560, 2560),
]

kernel_rows = []
for name, m, k, n in sizes:
    w_t, _ = quantize_ternary(torch.randint(-1, 2, (n, k), dtype=torch.float32))
    packed, n_pairs, n_padded = pack_tl1(w_t)
    x_i8 = torch.randint(-127, 128, (m, k), dtype=torch.int8)

    t_t = bench.Timer(
        stmt="lib.ternary_gemm(x, w, np, npairs)",
        globals={"lib": lib, "x": x_i8, "w": packed, "np": n_padded, "npairs": n_pairs},
    ).blocked_autorange(min_run_time=1.0)

    t_f = bench.Timer(
        stmt="torch.mm(x, w)",
        globals={"torch": torch, "x": torch.randn(m, k), "w": torch.randn(k, n)},
    ).blocked_autorange(min_run_time=1.0)

    kernel_rows.append((name, f"{m}x{k}x{n}", t_t.median * 1e6, t_f.median * 1e6))

print(f"{'name':<14} {'shape':<16} {'ternary us':>11} {'float us':>11} {'speedup':>8}")
print("-" * 64)
for name, shape, tt, tf in kernel_rows:
    print(f"{name:<14} {shape:<16} {tt:>11.1f} {tf:>11.1f} {tf / tt:>7.2f}x")

## 2. smelt CPU: pp512 / tg128

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
proc = psutil.Process()

gc.collect()
rss0 = proc.memory_info().rss
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
model.eval()
smelt.quantize(model)
smelt_mem = (proc.memory_info().rss - rss0) / 1e6
print(f"smelt RSS: {smelt_mem:.0f} MB")

# prefill: time to process PP tokens
prefill_ids = torch.randint(0, tokenizer.vocab_size, (1, PP))
with torch.no_grad():
    model(prefill_ids)  # warmup

t0 = time.perf_counter()
with torch.no_grad():
    model(prefill_ids)
t_prefill = time.perf_counter() - t0
smelt_pp = PP / t_prefill
print(f"smelt prefill (pp{PP}): {smelt_pp:.1f} tok/s")

# decode: generate TG tokens
prompt_ids = tokenizer.encode("The meaning of life is", return_tensors="pt")
with torch.no_grad():
    model.generate(prompt_ids, max_new_tokens=4, do_sample=False)  # warmup

t0 = time.perf_counter()
with torch.no_grad():
    out = model.generate(prompt_ids, max_new_tokens=TG, do_sample=False)
t_decode = time.perf_counter() - t0
n_gen = out.shape[1] - prompt_ids.shape[1]
smelt_tg = n_gen / t_decode
print(f"smelt decode  (tg{TG}): {smelt_tg:.1f} tok/s")
print(tokenizer.decode(out[0], skip_special_tokens=True)[:200])

del model
gc.collect()

## 3. HF float16 GPU (if available)

In [ ]:
gpu_pp = None
gpu_tg = None
gpu_mem = None

if HAS_CUDA:
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model_gpu = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.float16, device_map="auto"
    )
    model_gpu.eval()
    gpu_mem = torch.cuda.max_memory_allocated() / 1e6
    print(f"GPU VRAM: {gpu_mem:.0f} MB")

    prefill_gpu = prefill_ids.to(model_gpu.device)
    with torch.no_grad():
        model_gpu(prefill_gpu)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        model_gpu(prefill_gpu)
    torch.cuda.synchronize()
    gpu_pp = PP / (time.perf_counter() - t0)
    print(f"GPU prefill (pp{PP}): {gpu_pp:.1f} tok/s")

    prompt_gpu = prompt_ids.to(model_gpu.device)
    with torch.no_grad():
        model_gpu.generate(prompt_gpu, max_new_tokens=4, do_sample=False)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out_gpu = model_gpu.generate(prompt_gpu, max_new_tokens=TG, do_sample=False)
    torch.cuda.synchronize()
    n_gen = out_gpu.shape[1] - prompt_gpu.shape[1]
    gpu_tg = n_gen / (time.perf_counter() - t0)
    print(f"GPU decode  (tg{TG}): {gpu_tg:.1f} tok/s")

    del model_gpu
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("No GPU, skipping")

## 4. Int ops microbenchmark

In [ ]:
from smelt.plac import PLACFunc


def silu(x):
    return torch.nn.functional.silu(x.float()).to(x.dtype)


plac = PLACFunc(silu, -8, 8, target_mae=1e-2)

N = 6912
x_q16 = torch.randint(-8 * 65536, 8 * 65536, (1, N), dtype=torch.int32)
x_float = torch.randn(1, N)

int_ops = [
    ("int_rescale", "lib.int_rescale(x, 1311)", {"lib": lib, "x": x_q16}),
    (
        "plac_int32",
        "lib.plac_int32(x, bp, ic, s, e, ns)",
        {
            "lib": lib,
            "x": x_q16,
            "bp": plac._breakpoints,
            "ic": plac._intercepts,
            "s": plac._signs,
            "e": plac._exps,
            "ns": plac.n_segments,
        },
    ),
    ("int_quantize", "lib.int_quantize(x)", {"lib": lib, "x": x_q16}),
    ("int_mul", "lib.int_mul(a, b)", {"lib": lib, "a": x_q16, "b": x_q16}),
    ("torch.silu", "torch.nn.functional.silu(x)", {"torch": torch, "x": x_float}),
]

print(f"{'op':<16} {'time us':>10}")
print("-" * 28)
for name, stmt, globs in int_ops:
    t = bench.Timer(stmt=stmt, globals=globs).blocked_autorange(min_run_time=0.5)
    print(f"{name:<16} {t.median * 1e6:>10.1f}")

## 5. Bar plots

In [ ]:
methods = [f"smelt\n({cpu_name})"]
pp_speeds = [smelt_pp]
tg_speeds = [smelt_tg]
mems = [smelt_mem]
colors = ["#e74c3c"]

if gpu_tg is not None:
    methods.append(f"HF float16\n({gpu_label})")
    pp_speeds.append(gpu_pp)
    tg_speeds.append(gpu_tg)
    mems.append(gpu_mem)
    colors.append("#2ecc71")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, vals, title in [
    (axes[0], pp_speeds, f"Prefill (pp{PP}) tok/s"),
    (axes[1], tg_speeds, f"Decode (tg{TG}) tok/s"),
    (axes[2], mems, "Peak memory (MB)"),
]:
    bars = ax.barh(methods, vals, color=colors)
    for bar, v in zip(bars, vals, strict=True):
        ax.text(
            v + max(vals) * 0.02,
            bar.get_y() + bar.get_height() / 2,
            f"{v:.1f}",
            va="center",
            fontsize=10,
        )
    ax.set_title(title)
    ax.invert_yaxis()

plt.suptitle(f"BitNet b1.58 2B — {cpu_name}", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("benchmark.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n{'method':<28} {'pp tok/s':>10} {'tg tok/s':>10} {'mem MB':>10}")
print("-" * 60)
for m, pp, tg, mem in zip(methods, pp_speeds, tg_speeds, mems, strict=True):
    print(f"{m.replace(chr(10), ' '):<28} {pp:>10.1f} {tg:>10.1f} {mem:>10.0f}")